# 项目文件

## 说明

本次项目我使用了随机森林算法，通过分析歌曲的音乐特征来猜测其是否能成为热门歌曲，代码中的注释后续会逐渐完善

## 代码

### 训练模型

运行下面的代码框即可在目录中生成spotify_popularity_model.joblib模型文件

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score,confusion_matrix
from sklearn.ensemble import RandomForestClassifier
import joblib


df = pd.read_csv("songs_normalize.csv")
main_genres = ["pop","rock","hip hop","Dance/Electronic","country","metal","R&B","Folk/Acoustic"]
for g in main_genres:
  df[f"genre_{g}"] = 0
for idx,row in df.iterrows():
  genres = row["genre"].split(",")
  for g in genres:
    g_clean = g.strip().lower()
    if g_clean in main_genres:
      df.loc[idx,f"genre_{g_clean}"] = 1

#df["genre_main"] = df["genre"].apply(
#    lambda x:x if x in main_genres else "other"
#)
#genre_dummies = pd.get_dummies(df["genre_main"],prefix="genre")

df["energy_dance"] = df["energy"] * df["danceability"]

#df = pd.concat([df,genre_dummies],axis=1)
feature_cols = [
    "danceability","energy","valence","acousticness",
    "speechiness","instrumentalness","tempo","loudness","year",
    "genre_pop","genre_hip hop","genre_rock","genre_Dance/Electronic",
    "energy_dance","genre_country","genre_R&B","genre_metal","genre_Folk/Acoustic"
    ]

X = df[feature_cols]
popular_score = 70
y = np.where(df['popularity'] >= popular_score,1,0)
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.3,
    random_state=42,
    stratify=y
)

log_model = RandomForestClassifier(
    n_estimators=700,
    max_depth=20,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)
log_model.fit(X_train,y_train)

y_pred = log_model.predict(X_test)
print(pd.Series(y_pred).value_counts())

tn,fp,fn,tp = confusion_matrix(y_test,y_pred).ravel()
print("TN:",tn,"TP:",tp,"FN:",fn,"FP:",fp)
acc_normal = (tn+tp)/(tn+fp+fn+tp)
acc_balanced = balanced_accuracy_score(y_test,y_pred)
print("Normal accuracy:",round(acc_normal,4),
      "Balanced accuracy:",round(acc_balanced,4))

y_proba = log_model.predict_proba(X_test)[:,1]
thresholds = np.linspace(0.3,0.35,6000)


best_threshold = 0
best_bal_acc = 0
best_tn = best_fp = best_fn = best_tp = 0

for t in thresholds:
  pred = (y_proba>=t).astype(int)
  tn,fp,fn,tp = confusion_matrix(y_test,pred).ravel()
  bal_acc = balanced_accuracy_score(y_test,pred)

  if bal_acc > best_bal_acc:
    best_bal_acc = bal_acc
    best_threshold = t
    best_tn,best_fp,best_fn,best_tp = tn,fp,fn,tp

print(f"最佳阈值：{best_threshold}")
print(f"最高平衡准确率：{best_bal_acc}")
print(f"普通准确率：{(best_tp+best_tn)/(best_tn+best_fp+best_fn+best_tp)}")
print("TN:",best_tn,"TP:",best_tp,"FN:",best_fn,"FP:",best_fp)

joblib.dump(log_model,"spotify_popularity_model.joblib")
print("模型保存至spotify_popularity_model.joblib")

### 使用模型

运行以下代码，输入音频特征（可使用大模型得到特征值），模型会预测音乐热门的可能性

**（注：模型给出的意见仅为参考！）**

In [ ]:
import joblib
import pandas as pd
import numpy as np

model = joblib.load("spotify_popularity_model.joblib")

feature_cols = [
    "danceability","energy","valence","acousticness",
    "speechiness","instrumentalness","tempo","loudness","year",
    "genre_pop","genre_hip hop","genre_rock","genre_Dance/Electronic",
    "energy_dance","genre_country","genre_R&B","genre_metal","genre_Folk/Acoustic"
    ]


try:
    danceability = float(input("danceability: "))
    energy = float(input("energy: "))
    valence = float(input("valence: "))
    acousticness = float(input("acousticness: "))
    speechiness = float(input("speechiness: "))
    instrumentalness = float(input("instrumentalness: "))
    tempo = float(input("tempo: "))
    loudness = float(input("loudness: "))
    year = int(input("year (1999~2026): "))
    genre_input = input("genre: ").strip().lower()

    genre_pop = 0
    genre_hiphop = 0
    genre_rock = 0
    genre_dance = 0
    genre_country = 0
    genre_metal = 0
    genre_rb = 0
    genre_folk = 0

    genres = [g.strip() for g in genre_input.split(",")]

    for g in genres:
        if g == "pop":
            genre_pop = 1
        elif g == "hip hop":
            genre_hiphop = 1
        elif g == "rock":
            genre_rock = 1
        elif g in ["dance/electronic", "dance", "electronic"]:
            genre_dance = 1
        elif g == "country":
            genre_country = 1
        elif g == "metal":
            genre_metal = 1
        elif g == "r&b":
            genre_rb = 1
        elif g in ["folk/acoustic","folk","acoustic"]:
            genre_folk = 1

    energy_dance = energy * danceability

    input_data = pd.DataFrame([[
           danceability, energy, valence, acousticness,
            speechiness, instrumentalness, tempo, loudness, year,
           genre_pop, genre_hiphop, genre_rock, genre_dance,
           energy_dance, genre_country, genre_rb, genre_metal,genre_folk
           ]], columns=feature_cols)


    prob = model.predict_proba(input_data)[0][1]
    pred_class = model.predict(input_data)[0]

    print(f"热门概率：{prob:.2%}")
    if pred_class == 1:
      print("很有可能成为热门歌曲")
    else:
      print("很有可能不会成为热门歌曲")
except ValueError as e:
  print(f"\n报错：{e}")
except Exception as e:
  print(f"\n报错：{e}")

## 小记

我本来在后面还写了一块从[Spotify开发者平台](https://developer.spotify.com/dashboard)调取歌曲audio_feature的代码，但是尝试了好几次都无法使用。

后来才在开发者文档中了解到此项功能已在2024年11月废除公共API接口，为此我深感抱歉。